In [1]:
!pip install timm pycocoevalcap evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 17.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [2]:
import os
import io
import glob
import random
import hashlib
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader

import timm
import nltk
import evaluate

from datasets import load_dataset, Dataset, DatasetDict
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

print('Imports complete')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Imports complete
Using device: cuda


In [3]:
print("Loading Hugging Face dataset...")
dataset = load_dataset("Sandesh-Lav/8kpotato-tuber-caption-dataset")

train_split = dataset["train"]
val_split = dataset["validation"]

print("Train split:", len(train_split))
print("Validation split:", len(val_split))

from datasets import concatenate_datasets

full_ds = concatenate_datasets([train_split, val_split])

print("Combined dataset size:", len(full_ds))

Loading Hugging Face dataset...


DatasetNotFoundError: Dataset 'Sandesh-Lav/8kpotato-tuber-caption-dataset' doesn't exist on the Hub or cannot be accessed.

In [ ]:
def build_groups(ds):
    def image_hash(img_dict):
        return hashlib.md5(img_dict['bytes']).hexdigest()

    groups = defaultdict(list)

    for idx, ex in enumerate(ds):
        h = image_hash(ex['image'])
        groups[h].append(idx)

    return list(groups.values())

print('Grouping images...')
all_groups = build_groups(full_ds)
print('Total grouped samples:', len(all_groups))

def create_generator(ds, groups):
    def gen():
        for group in groups:
            first_idx = group[0]
            image_data = ds[first_idx]['image']
            caps = [ds[idx]['caption'] for idx in group]
            yield {
                'image': image_data,
                'captions': caps
            }
    return gen

processed_full_ds = Dataset.from_generator(create_generator(full_ds, all_groups))
print('Processed dataset size:', len(processed_full_ds))

In [ ]:
initial_split = processed_full_ds.train_test_split(test_size=0.2, seed=42)
val_test_split = initial_split["test"].train_test_split(test_size=0.5, seed=42)

train_groups = initial_split["train"]
val_groups = val_test_split["train"]
test_groups = val_test_split["test"]

final_dataset = DatasetDict({
    "train": train_groups,
    "validation": val_groups,
    "test": test_groups
})

print("Train:", len(train_groups))
print("Validation:", len(val_groups))
print("Test:", len(test_groups))

In [ ]:
PAD_TOKEN, PAD_IDX = "<PAD>", 0
SOS_TOKEN, SOS_IDX = "<SOS>", 1
EOS_TOKEN, EOS_IDX = "<EOS>", 2
UNK_TOKEN, UNK_IDX = "<UNK>", 3

word2idx = {
    PAD_TOKEN: PAD_IDX,
    SOS_TOKEN: SOS_IDX,
    EOS_TOKEN: EOS_IDX,
    UNK_TOKEN: UNK_IDX
}

idx2word = {
    PAD_IDX: PAD_TOKEN,
    SOS_IDX: SOS_TOKEN,
    EOS_IDX: EOS_TOKEN,
    UNK_IDX: UNK_TOKEN
}

word_counts = Counter()

print("Building vocabulary...")
for item in tqdm(train_groups):
    for caption in item["captions"]:
        words = caption.lower().split()
        word_counts.update(words)

idx = len(word2idx)

for word, count in word_counts.items():
    word2idx[word] = idx
    idx2word[idx] = word
    idx += 1

vocab_size = len(word2idx)

print("Vocabulary size:", vocab_size)

In [ ]:
MAX_LEN = 30
BATCH_SIZE = 32

image_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def load_clean_image(img_data):
    if isinstance(img_data, dict) and "bytes" in img_data:
        return Image.open(io.BytesIO(img_data["bytes"])).convert("RGB")
    return img_data.convert("RGB")

def tokenize_and_pad(caption_string, max_len=MAX_LEN):
    words = caption_string.lower().split()

    tokens = [word2idx.get(w, UNK_IDX) for w in words[:max_len - 2]]

    tokens = [SOS_IDX] + tokens + [EOS_IDX]

    padding_length = max_len - len(tokens)
    tokens += [PAD_IDX] * padding_length

    return torch.tensor(tokens, dtype=torch.long)

print("Transforms and tokenizer ready")

In [ ]:
def train_collate_fn(batch):
    images = []
    tokenized_caps = []

    for item in batch:
        img = load_clean_image(item["image"])
        images.append(image_transform(img))

        cap_str = random.choice(item["captions"])
        tokenized_caps.append(tokenize_and_pad(cap_str))

    return torch.stack(images), torch.stack(tokenized_caps)

def eval_collate_fn(batch):
    images = []
    raw_captions = []
    tokenized_caps = []

    for item in batch:
        img = load_clean_image(item["image"])
        images.append(image_transform(img))

        raw_captions.append(item["captions"])
        tokenized_caps.append(tokenize_and_pad(item["captions"][0]))

    return torch.stack(images), raw_captions, torch.stack(tokenized_caps)

train_loader = DataLoader(
    train_groups,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_collate_fn
)

val_loader = DataLoader(
    val_groups,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=eval_collate_fn
)

test_loader = DataLoader(
    test_groups,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=eval_collate_fn
)

print("DataLoaders created successfully")

In [ ]:
class ViTEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True
        )

        self.vit.head = nn.Identity()

        for p in self.vit.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.vit(x)

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_layers=4, nhead=8):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(MAX_LEN, embed_dim)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=2048,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(embed_dim, vocab_size)

    def generate_mask(self, size):
        return torch.triu(
            torch.ones(size, size, device=device),
            diagonal=1
        ).bool()

    def forward(self, features, captions):
        seq_len = captions.size(1)

        positions = torch.arange(
            0,
            seq_len,
            device=device
        ).unsqueeze(0).expand(captions.size(0), seq_len)

        x = self.embedding(captions) + self.pos_embedding(positions)

        memory = features.unsqueeze(1)

        tgt_mask = self.generate_mask(seq_len)

        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask
        )

        return self.fc(out)

class CaptionModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.encoder = ViTEncoder()
        self.decoder = TransformerDecoder(vocab_size)

    def forward(self, images, captions):
        features = self.encoder(images)
        return self.decoder(features, captions)

print("Initializing ViT model...")
model = CaptionModel(vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5
)

print("Model initialized successfully")

In [ ]:
def validate():
    model.eval()

    total_loss = 0

    with torch.no_grad():
        for imgs, raw_caps, tokenized_caps in tqdm(
            val_loader,
            desc="Validation"
        ):
            imgs = imgs.to(device)
            caps = tokenized_caps.to(device)

            outputs = model(imgs, caps[:, :-1])

            loss = criterion(
                outputs.reshape(-1, vocab_size),
                caps[:, 1:].reshape(-1)
            )

            total_loss += loss.item()

    avg_loss = total_loss / len(val_loader)

    print("Validation Loss:", avg_loss)

    return avg_loss

In [ ]:
EPOCHS = 45

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0

    print(f"\nStarting Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, (imgs, caps) in enumerate(
        tqdm(train_loader, desc=f"Epoch {epoch+1} Training")
    ):
        imgs = imgs.to(device)
        caps = caps.to(device)

        optimizer.zero_grad()

        outputs = model(imgs, caps[:, :-1])

        loss = criterion(
            outputs.reshape(-1, vocab_size),
            caps[:, 1:].reshape(-1)
        )

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if batch_idx % 20 == 0:
            print(
                f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}"
            )

    avg_train_loss = train_loss / len(train_loader)

    val_loss = validate()

    print(f"\nEpoch {epoch+1} Complete")
    print("Train Loss:", avg_train_loss)
    print("Val Loss:", val_loss)

In [ ]:
def generate_caption_from_image(image_tensor, max_len=MAX_LEN):
    model.eval()

    generated = [SOS_IDX]

    with torch.no_grad():
        image_tensor = image_tensor.unsqueeze(0).to(device)

        features = model.encoder(image_tensor)

        for _ in range(max_len):
            caption_tensor = torch.tensor(
                [generated],
                dtype=torch.long
            ).to(device)

            outputs = model.decoder(features, caption_tensor)

            next_token = outputs[:, -1, :].argmax(dim=-1).item()

            if next_token == EOS_IDX:
                break

            generated.append(next_token)

    words = [
        idx2word.get(idx, UNK_TOKEN)
        for idx in generated[1:]
    ]

    return words

In [ ]:
def evaluate():
    model.eval()

    bleu1, bleu2, bleu3, bleu4, bleu5 = [], [], [], [], []
    meteor_scores = []
    rouge_scores = []

    scorer = rouge_scorer.RougeScorer(
        ["rougeL"],
        use_stemmer=True
    )

    cider_scorer = Cider()

    gts = {}
    res = {}
    img_id = 0

    print("Starting evaluation...")

    with torch.no_grad():
        for imgs, raw_caps_list, _ in tqdm(
            test_loader,
            desc="Evaluating Metrics"
        ):
            for i in range(len(imgs)):
                pred = generate_caption_from_image(imgs[i])

                pred_sentence = " ".join(pred)

                ref_sentences = raw_caps_list[i]

                ref_lists = [ref.split() for ref in ref_sentences]

                smooth = SmoothingFunction().method1

                bleu1.append(
                    sentence_bleu(
                        ref_lists,
                        pred,
                        weights=(1,0,0,0,0),
                        smoothing_function=smooth
                    )
                )

                bleu2.append(
                    sentence_bleu(
                        ref_lists,
                        pred,
                        weights=(0.5,0.5,0,0,0),
                        smoothing_function=smooth
                    )
                )

                bleu3.append(
                    sentence_bleu(
                        ref_lists,
                        pred,
                        weights=(0.33,0.33,0.33,0,0),
                        smoothing_function=smooth
                    )
                )

                bleu4.append(
                    sentence_bleu(
                        ref_lists,
                        pred,
                        weights=(0.25,0.25,0.25,0.25,0),
                        smoothing_function=smooth
                    )
                )

                bleu5.append(
                    sentence_bleu(
                        ref_lists,
                        pred,
                        weights=(0.2,0.2,0.2,0.2,0.2),
                        smoothing_function=smooth
                    )
                )

                primary_ref = ref_sentences[0]

                meteor_scores.append(
                    meteor_score(
                        [primary_ref.split()],
                        pred_sentence.split()
                    )
                )

                rouge_scores.append(
                    scorer.score(
                        primary_ref,
                        pred_sentence
                    )["rougeL"].fmeasure
                )

                gts[img_id] = ref_sentences
                res[img_id] = [pred_sentence]

                img_id += 1

    cider_score, _ = cider_scorer.compute_score(gts, res)

    print("\n===== FINAL METRICS =====")
    print("BLEU-1:", np.mean(bleu1))
    print("BLEU-2:", np.mean(bleu2))
    print("BLEU-3:", np.mean(bleu3))
    print("BLEU-4:", np.mean(bleu4))
    print("BLEU-5:", np.mean(bleu5))
    print("METEOR:", np.mean(meteor_scores))
    print("ROUGE:", np.mean(rouge_scores))
    print("CIDEr:", cider_score)

In [ ]:
evaluate()